In [ ]:
# sklearn basic models, aiming to predict DO from other indicators, but only on the old data from Dr. Kellogg (through only 2015)

In [ ]:
# importing modules and packages 
import pandas as pd 
import matplotlib.pyplot as plt 
import seaborn as sns 
from sklearn.model_selection import train_test_split 
from sklearn.metrics import mean_squared_error, mean_absolute_error 
from sklearn.preprocessing import OneHotEncoder
from sklearn import linear_model
from sklearn.tree import DecisionTreeRegressor

In [ ]:
df = pd.read_csv('../../data/fixed_site_data.csv')

In [ ]:
print(df.dtypes)

# date to datetime (though we change it again after so idk if this is even smart)
df['Date'] = pd.to_datetime(df['Date'])
df_pivot = df.pivot_table(index=['Date', 'site', 'depth'], columns='param', values='measure', aggfunc='mean')
df_pivot.reset_index(inplace=True)

# get all BR site
br_data = df_pivot[df_pivot['site'] == 'BR']
br_data

In [ ]:
# only surface sondes
br_data_surface = br_data[br_data['depth'] == "surface"]
br_data_surface.isnull().sum()

In [ ]:
br_data_surface.drop(columns=['site', 'depth', "Density", "SpCond", "Turb", "DO.", "Depth"], inplace=True)
br_data_surface.dropna(subset=['DO.Conc'], inplace=True)
br_data_surface

In [ ]:
br_data_surface.fillna(method='ffill', inplace=True)
#br_data_surface.drop(columns=['Date'], inplace=True)

# turn date to ordinal
br_data_surface['Date'] = br_data_surface['Date'].map(pd.Timestamp.toordinal)
br_data_surface

In [ ]:
# reset index
br_data_surface.reset_index(drop=True, inplace=True)
br_data_surface

In [ ]:
X = br_data_surface.drop(columns=['DO.Conc'])
y = br_data_surface['DO.Conc']

print(X)
print(y)

In [ ]:
X_train, X_test, y_train, y_test = train_test_split( 
    X, y, test_size=0.3, random_state=101) 

In [ ]:
classifiers = [
    linear_model.LinearRegression(),
    linear_model.Ridge(alpha=0.1),
    linear_model.Lasso(),
    linear_model.ElasticNet(),
    linear_model.BayesianRidge(),
    linear_model.ARDRegression(),
    #linear_model.LogisticRegression()
    # apparently this is categorical
    linear_model.TheilSenRegressor(),
    linear_model.HuberRegressor(),
    linear_model.PassiveAggressiveRegressor(),
    linear_model.RANSACRegressor(),
    linear_model.RidgeCV(alphas=[0.01, 0.1, 1.0, 10.0, 100.0], store_cv_results=True),
    # decision tree
    DecisionTreeRegressor(criterion='squared_error', max_depth=None, min_samples_split=2, min_samples_leaf= 8)
    ]

In [ ]:
for item in classifiers:
    print(item)
    #print(item.get_params())
    clf = item
    clf.fit(X_train, y_train)
    print('mean_absolute_error : ', mean_absolute_error(y_test, clf.predict(X_test)))
    print('r_squared : ', clf.score(X_test, y_test))
    try: 
        print(clf.alpha_)
    except:
        pass
    print('-----------------------------------')

In [ ]:
from sklearn.model_selection import GridSearchCV, RandomizedSearchCV
from sklearn.neural_network import MLPRegressor
from sklearn.preprocessing import StandardScaler

In [ ]:
# tree hyperparameters

param_grid = {
    'max_depth': [3, 5, 10, None],
    'min_samples_split': [2, 5, 10],
    'min_samples_leaf': [1, 2, 4, 8]
}

grid_search = GridSearchCV(DecisionTreeRegressor(random_state=42), param_grid, cv=5, scoring='r2')
grid_search.fit(X_train, y_train)

print("Best parameters:", grid_search.best_params_)

In [ ]:
scaler = StandardScaler()
X_train = scaler.fit_transform(X_train)
X_test = scaler.transform(X_test)

In [ ]:
regr = MLPRegressor(random_state=1, max_iter=2000, hidden_layer_sizes=(50, 50, 50), activation='relu', solver='sgd', alpha=0.05)
regr.fit(X_train, y_train)

In [ ]:
y_pred = regr.predict(X_test)
mse = mean_squared_error(y_test, y_pred)
r2 = regr.score(X_test, y_test)
mae = mean_absolute_error(y_test, y_pred)

print(f"Mean Squared Error: {mse:.2f}")
print(f"Mean Absolute Error: {mae:.2f}")
print(f"R² Score: {r2:.2f}")

In [ ]:
# MLP hyperparameters

param_grid = {
    'hidden_layer_sizes': [(50,50,50), (50,100,50), (100,)],
    'activation': ['tanh', 'relu'],
    'solver': ['sgd', 'adam'],
    'alpha': [0.0001, 0.05],
    'learning_rate': ['constant','adaptive'],
}

grid_search = GridSearchCV(MLPRegressor(max_iter=2000, random_state=101), param_grid, cv=5, scoring='r2', verbose=2)
# grid_search.fit(X_train, y_train)

random_search = RandomizedSearchCV(MLPRegressor(max_iter=2000, random_state=101), param_distributions=param_grid, n_iter=10, cv=5, scoring='r2', verbose=2)
random_search.fit(X_train, y_train)

print("Best parameters:", random_search.best_params_)


In [ ]:
# visualize weights

weights = regr.coefs_
weights

In [ ]:
br_data_surface